# Reproduce dataset sampling and experimental partitions

Companion to `bert-retrain-each-round-final.ipynb`. Runs on CPU; no model training or GPU is needed.

**What is reproduced:** DynaHate and Davidson majority-class undersampling, label mappings, outer partitions, and the internal fitting/stopping/calibration partitions for repetition seeds 42, 43, and 44. The sampling and split operations are copied from the experiment notebook, preserving their order.

**Mody input:** The experiment reads an existing `balanced_data.csv` with `Content` and `Label` columns. It does not contain the procedure that created that file. Supply the same input to reproduce the reported Mody partitions. This notebook does not invent a raw-Mody sampling procedure. Its Mody source-row IDs refer to that supplied CSV, not the original full corpus.

## Run instructions
1. Install the dependencies below if needed.
2. Set `ORIGINAL_CSV` to the same Mody CSV used for training. On Kaggle, attach it as an input. Set `DATASETS` to the two public datasets if it is unavailable.
3. Run all cells. Public CSV downloads are cached locally; explicit input paths take priority.
4. Publish the notebook and, where permitted, the generated manifests and text-free split files. Processed text CSVs stay local by default. Verify redistribution terms before uploading dataset content.

Source-row IDs are zero-based CSV data-record positions, excluding the header. They are meaningful only together with the source-file SHA-256. Existing row-based splits are reproduced exactly; duplicate text is audited, not removed or regrouped.


In [ ]:
%pip install numpy pandas scikit-learn

In [ ]:
import hashlib, json, re, platform, urllib.request, importlib.metadata
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

DATASETS = ['original', 'dynahate', 'davidson']
ORIGINAL_CSV = None  # Set to the experiment's balanced_data.csv; otherwise auto-discover.
DYNAHATE_CSV = None
DAVIDSON_CSV = None
SPLIT_SEED = 42
REPEAT_SEEDS = [42, 43, 44]
OUTPUT = Path('/kaggle/working/data_preparation') if Path('/kaggle').exists() else Path.cwd()/'data_preparation'
CACHE = OUTPUT/'source_cache'
EXPORT_TEXT = False  # Optional local prepared CSVs; not required for publishing split assignments.
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

SOURCES = {}
def find_csv(explicit, names):
    if explicit:
        p = Path(explicit)
        if not p.is_file(): raise FileNotFoundError(p)
        return p
    found = []
    for root in [Path('/kaggle/input'), Path('/content'), Path.cwd()/'data']:
        if root.exists():
            found.extend(p for p in root.rglob('*.csv') if p.name.lower() in names)
    if len(found) > 1: raise ValueError(f'Multiple CSVs found; set the input path explicitly: {found}')
    return found[0] if found else None

# Keep the original loader unchanged apart from tracking source rows.
# Cache downloads and record the exact bytes passed to pandas.
_read_csv = pd.read_csv
def cached_read_csv(source, *args, **kwargs):
    source_string = str(source)
    if source_string.startswith('https://'):
        p = CACHE/(hashlib.sha256(source_string.encode()).hexdigest()[:16] + '.csv')
        if not p.exists():
            temporary = p.with_suffix('.tmp')
            urllib.request.urlretrieve(source_string, temporary)
            temporary.replace(p)
    else:
        p = Path(source)
    frame = _read_csv(p, *args, **kwargs)
    SOURCES[source_string] = {'source': source_string, 'sha256': file_hash(p), 'source_rows': len(frame)}
    return frame


In [ ]:
def balanced(df, seed=42):
    n=int(df.y.value_counts().min())
    return pd.concat([g.sample(n=n,random_state=seed) for _,g in df.groupby('y')]).sample(frac=1,random_state=seed).reset_index(drop=True)

def load_data(name):
    if name=='original':
        p=find_csv(ORIGINAL_CSV, {'balanced_data.csv'})
        if p is None: raise FileNotFoundError('Attach balanced_data.csv or set ORIGINAL_CSV; public datasets can run without it.')
        raw=cached_read_csv(p)
        raw['source_row_id']=np.arange(len(raw))
        raw=raw.dropna(subset=['Content','Label']).reset_index(drop=True)
        encoder=LabelEncoder().fit(raw.Label)
        if len(encoder.classes_)!=2: raise ValueError('Original corpus must have exactly two labels.')
        df=pd.DataFrame({'text':raw.Content.astype(str), 'y':encoder.transform(raw.Label), 'source_row_id':raw.source_row_id.values})
        names=[str(x) for x in encoder.classes_]; source=str(p)
    elif name=='dynahate':
        url='https://raw.githubusercontent.com/bvidgen/Dynamically-Generated-Hate-Speech-Dataset/main/Dynamically%20Generated%20Hate%20Dataset%20v0.2.3.csv'
        p=find_csv(DYNAHATE_CSV, {'dynamically generated hate dataset v0.2.3.csv','dynahate.csv'})
        raw=cached_read_csv(p or url)
        raw['source_row_id']=np.arange(len(raw))
        tc=next((c for c in raw if c.lower() in {'text','content','comment','tweet','sentence'}),None)
        lc=next((c for c in raw if c.lower() in {'label','class','target','gold_label'}),None)
        if tc is None or lc is None: raise ValueError(f'Unrecognized DynaHate columns: {raw.columns.tolist()}')
        raw=raw.dropna(subset=[tc,lc])
        mapping={'hate':1,'hateful':1,'1':1,'1.0':1,'yes':1,'true':1,
                 'nothate':0,'not_hate':0,'not-hate':0,'not hate':0,'non-hate':0,'0':0,'0.0':0,'no':0,'false':0}
        labels=raw[lc].astype(str).str.strip().str.lower()
        if not set(labels).issubset(mapping): raise ValueError(f'Unrecognized DynaHate labels: {set(labels)-set(mapping)}')
        df=balanced(pd.DataFrame({'text':raw[tc].astype(str).values,'y':labels.map(mapping).values, 'source_row_id':raw.source_row_id.values}))
        names=['Not hate','Hate']; source=str(p or url)
    elif name=='davidson':
        url='https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv'
        p=find_csv(DAVIDSON_CSV, {'labeled_data.csv','davidson.csv'})
        raw=cached_read_csv(p or url)
        raw['source_row_id']=np.arange(len(raw))
        raw=raw.dropna(subset=['tweet','class'])
        cls=raw['class'].astype(int)
        if not set(cls).issubset({0,1,2}): raise ValueError('Unexpected Davidson class values.')
        df=balanced(pd.DataFrame({'text':raw.tweet.astype(str).values,'y':(cls!=2).astype(int).values, 'source_row_id':raw.source_row_id.values}))
        names=['Neither','Offensive or hateful']; source=str(p or url)
    else: raise ValueError(name)
    df=df.reset_index(drop=True); df['row_id']=np.arange(len(df))
    return df, {'source':source,'class_names':names,'retained_rows':len(df),
                'data_hash':digest(df[['text','y']].values.tolist())}

def clean_text(text):
    t=re.sub(r'http\S+|www\.\S+', ' ', str(text))
    t=re.sub(r'@\w+', ' @user ', t)
    return re.sub(r'\s+', ' ', t).strip() or '[empty]'

def split_data(df, seed=42):
    dev,test=train_test_split(df,test_size=.20,stratify=df.y,random_state=seed)
    annotated,unl=train_test_split(dev,test_size=.80,stratify=dev.y,random_state=seed)
    train,val=train_test_split(annotated,test_size=.20,stratify=annotated.y,random_state=seed)
    return {k:v.reset_index(drop=True) for k,v in {'train':train,'val':val,'unl':unl,'test':test}.items()}

def overlap_audit(parts):
    groups={k:set(v.text.map(clean_text).str.casefold()) for k,v in parts.items()}
    return {f'{a}__{b}':len(groups[a]&groups[b]) for a,b in combinations(groups,2)}

## Prepare and export

Outer partition labels are `train` (L), `val` (V), `unl` (U), and `test` (T). Internal roles split only L, matching the development phase used by the iterative experiment. U reference labels are not exported in the split-assignment files. The optional prepared text file contains reference labels for retrospective evaluation; never pass U labels to the training procedure.

Source files are cached by URL. To use a different revision, supply an explicit local CSV; file hashes identify exactly which inputs were used. A cached download alone does not pin the public source version used in the original experiment.


In [ ]:
summaries = []
RAW, PARTS = {}, {}
for name in DATASETS:
    df, meta = load_data(name)
    parts = split_data(df, SPLIT_SEED)
    RAW[name], PARTS[name] = df, parts
    before = dict(SOURCES[meta['source']])
    assignments = pd.concat([
        frame[['row_id', 'source_row_id']].assign(partition=partition)
        for partition, frame in parts.items()
    ], ignore_index=True)
    assert assignments.row_id.is_unique
    assert set(assignments.row_id) == set(df.row_id)
    assignments.to_csv(OUTPUT/f'{name}_splits.csv', index=False)
    internal = []
    for seed in REPEAT_SEEDS:
        lab = parts['train']
        remaining, cal = train_test_split(np.arange(len(lab)), test_size=0.10,
            stratify=lab.y, random_state=seed)
        fit, stop = train_test_split(remaining, test_size=0.10/(1-0.10),
            stratify=lab.y.values[remaining], random_state=seed+1)
        for role, positions in [('fit', fit), ('stop', stop), ('cal', cal)]:
            internal.append(lab.iloc[positions][['row_id', 'source_row_id']].assign(repeat_seed=seed, role=role))
    pd.concat(internal, ignore_index=True).to_csv(OUTPUT/f'{name}_internal_roles.csv', index=False)
    class_counts = {partition: {str(int(k)): int(v) for k, v in frame.y.value_counts().sort_index().items()}
                    for partition, frame in parts.items()}
    meta.update(source_file=before, sampling_seed=42 if name != 'original' else None,
        sampling='prebalanced input; upstream sampling unavailable' if name == 'original' else 'equal-class undersampling',
        split_seed=SPLIT_SEED, repetition_seeds=REPEAT_SEEDS,
        source_row_definition='zero-based CSV data-record position before dropping missing rows',
        selected_source_rows=[int(x) for x in df.source_row_id],
        splits={k: [int(x) for x in v.row_id] for k, v in parts.items()},
        class_counts=class_counts, normalized_text_overlap=overlap_audit(parts))
    write_json(OUTPUT/f'{name}_manifest.json', meta)
    if EXPORT_TEXT:
        df.to_csv(OUTPUT/f'{name}_prepared.csv', index=False)
    for partition, frame in parts.items():
        summaries.append({'dataset': name, 'partition': partition, 'rows': len(frame),
                          'class_0': int((frame.y == 0).sum()), 'class_1': int((frame.y == 1).sum())})
    print(name, '| normalized text overlap:', meta['normalized_text_overlap'])
    if any(meta['normalized_text_overlap'].values()):
        print('Repeated text crosses partitions. These files reproduce the experiment, not duplicate-grouped confirmation splits.')

summary = pd.DataFrame(summaries)
summary.to_csv(OUTPUT/'partition_counts.csv', index=False)
write_json(OUTPUT/'environment.json', {'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in ['numpy','pandas','scikit-learn']}})
display(summary)
print('Output folder:', OUTPUT.resolve())


## GitHub notes

Publish this notebook beside the training notebook. A README should identify the dataset sources and citations, specify the required Mody input, and explain how the data-preparation and training notebooks connect. This companion does not modify the training notebook: that notebook continues to use its own identical sampling and splitting functions.

The manifests identify the source-file hashes, sampled source rows, partition membership, class mappings, and duplicate overlaps. Archive the exact source revision or checksum alongside a release to prevent silent changes in upstream data. The split files and internal-role files preserve row order within each partition/role.

For Mody, include the original preparation script or documented sampled identifiers if available. Until then, describe reproduction as conditional on access to the same `balanced_data.csv`; do not claim raw-corpus-to-results reproducibility.
